# 03 · Evaluation

Scores every trained model on a **separate test zip** (different room or day). The models
receive the image and the meta vector only; LiDAR depth is used solely as the truth to score
against.

Compared here:
- every checkpoint in `RUNS` (by default the U-Net with focus, the image-only ablation and the
  DINOv2 model),
- a no-learning baseline that takes the calibrated focus distance as the object's distance.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
# checkpoint name -> label in the tables; the first one is the reference model
RUNS = {"model": "U-Net + focus", "model_image_only": "image only", "model_embed": "U-Net + DINOv2"}

In [ ]:
from unetultra.calibration import LensCalibration
from unetultra.dataset import SessionDataset
from unetultra.models import uses_meta
from unetultra.session import find_sessions
from unetultra.train import load_model

TEST_ZIPS = find_sessions(DATA, "test")        # only sessions marked split: test
assert TEST_ZIPS, "no test session found in ../data"
calibration = LensCalibration.load(MODELS / "calibration.json")
RUNS = {name: label for name, label in RUNS.items() if (MODELS / f"{name}.pt").exists()}
models, test_sets = {}, {}
for name, label in RUNS.items():
    models[label], checkpoint = load_model(MODELS / f"{name}.pt")
    test_sets[label] = SessionDataset(TEST_ZIPS, calibration, use_meta=uses_meta(checkpoint["architecture"]))
    print(f"{label}: {checkpoint['architecture']}, checkpoint from epoch {checkpoint['epoch']}")
reference = next(iter(RUNS.values()))
test_set = test_sets[reference]
print(f"{len(test_set)} test samples from {len(TEST_ZIPS)} session(s)")

## Metrics

In [ ]:
import json
from unetultra import metrics
from unetultra.export import headline_metrics

results, summaries, predictions = {}, {}, {}
for (name, label) in RUNS.items():
    rows, image_rows, predictions[label] = metrics.evaluate_model(models[label], test_sets[label])
    results[label], summaries[label] = rows, metrics.summarise(rows, image_rows)
    # Kept next to the checkpoint so the export can copy them into the model card.
    (MODELS / f"{name}_test_metrics.json").write_text(json.dumps(headline_metrics(summaries[label])))
rows, summary = results[reference], summaries[reference]
print(metrics.format_summary(summaries))

### Focus-only baseline

Using the calibrated focus distance as the distance of the object the lens focused on, with no
model at all. If a model cannot beat this, it has learned nothing about distance beyond focus.

In [ ]:
baseline = summary["distance_focus_baseline"]
print(f"focused objects: {baseline['n']}")
print(f"  {'focus distance alone':22}: {baseline['mae']:.2f} cm mean abs error ({baseline['mape']:.1f}%)")
for label, s in summaries.items():
    on_targets = s["distance_model_on_focus_targets"]
    print(f"  {label:22}: {on_targets['mae']:.2f} cm mean abs error ({on_targets['mape']:.1f}%)")

### Size: before and after the bias fix

The reference method (2nd–98th percentile between pixel centres) under-reports size by about 4%
and is thrown off by depth outliers at the object's edge. The fix measures outlier pixels at
the object's median depth and uses the full pixel footprint.

In [ ]:
print(metrics.format_size_fix(summary))
lidar_vs_ruler = summary["lidar_width_vs_measured"]
if lidar_vs_ruler["n"]:
    print(f"\nFor reference, LiDAR-derived width against the ruler: {lidar_vs_ruler['mae']:.2f} cm "
          f"({lidar_vs_ruler['mape']:.1f}%) over {lidar_vs_ruler['n']} objects")

### By distance

In [ ]:
for label, r in results.items():
    print(label)
    for row in metrics.distance_error_by_bin(r):
        print(f"  {row['bin_cm'][0]:>3}-{row['bin_cm'][1]:<3} cm  n={row['n']:<3} "
              f"mean abs {row['mae']:.2f} cm ({row['mape']:.1f}%)  bias {row['bias']:+.2f} cm")

## Plots

In [ ]:
from unetultra import plots

plots.predicted_vs_true_distance(results)
plots.error_vs_distance(results)
plots.error_vs_distance(results, key="pred_width_cm", truth="true_width_cm", ylabel="width error (cm)")
plots.error_vs_lens(rows);

In [ ]:
plots.example_overlays(test_set, predictions[reference], rows);

## Verdict

Generated from the numbers above. Add your own reading underneath after looking at the plots.

In [ ]:
if "image only" in summaries and reference != "image only":
    print(metrics.verdict(summary, summaries["image only"], rows))
for label, s in summaries.items():
    if label not in (reference, "image only") and s.get("distance", {}).get("n"):
        print(f"{label} against {reference}: distance error {s['distance']['mae']:.2f} cm vs {summary['distance']['mae']:.2f} cm, "
              f"Dice {s['dice']:.3f} vs {summary['dice']:.3f}.")

_Your notes:_